In [1]:
from transformers import BertTokenizer, BertModel
import torch

In [2]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
# Define the sentences for testing
sentences = [
    "I'm sitting in the bank to meet my financial advisor.",
    "I'm sitting in the bank, wathcing the flow of the river.",
    "I'm sitting in the bank, waiting for my number to be called."
]

# Define function to calculate consine similarity
def cosine_similarity(embedding1, embedding2):
  cos_sim = torch.nn.functional.cosine_similarity(embedding1.unsqueeze(0), embedding2.unsqueeze(0))
  return cos_sim.item()

# Define function to get the embedding of the search word in a sentence
def get_embedding(sentence, word):
  inputs = tokenizer(sentence, return_tensors="pt")

  token_ids = inputs["input_ids"]
  attention_mask = inputs["attention_mask"]

  with torch.no_grad():
    outputs = model(token_ids, attention_mask = attention_mask)

  last_hidden_states = outputs.last_hidden_state

  word_index = (token_ids == tokenizer.convert_tokens_to_ids(word)).nonzero(as_tuple=True)[1][0]

  word_embedding = last_hidden_states[0, word_index, :]

  return word_embedding

In [8]:
# Get embeddings for 'bank' in both sentences

search_word = "bank"
embeddings = [get_embedding(sentence, search_word) for sentence in sentences]

In [10]:
for i in range(len(sentences)):
  print(f"Sentence {i+1}: {sentences[i]}")
print()

Sentence 1: I'm sitting in the bank to meet my financial advisor.
Sentence 2: I'm sitting in the bank, wathcing the flow of the river.
Sentence 3: I'm sitting in the bank, waiting for my number to be called.



In [12]:
print(f"Cosine similarity between the embeddings of '{search_word}' from sentences 1 and 2 : ", cosine_similarity(embeddings[0], embeddings[1]))
print(f"Cosine similarity between the embeddings of '{search_word}' from sentences 1 and 3 : ", cosine_similarity(embeddings[0], embeddings[2]))
print(f"Cosine similarity between the embeddings of '{search_word}' from sentences 2 and 3 : ", cosine_similarity(embeddings[1], embeddings[2]))

Cosine similarity between the embeddings of 'bank' from sentences 1 and 2 :  0.5747623443603516
Cosine similarity between the embeddings of 'bank' from sentences 1 and 3 :  0.8928871750831604
Cosine similarity between the embeddings of 'bank' from sentences 2 and 3 :  0.6202016472816467
